# Image and Post Content Creator Chatbot

Given a **topic/theme**, this project:

1. 🎨 Generates a related **image** (gpt-image-1-mini),
2. ✍️ Generates a **short, catchy caption** to overlay on the image,
3. 📖 Generates an inspiring **quote from a book**,

and combines all three into a single image, ready to share on social media.

It brings together everything we learned in Week 2: **structured output**, **image generation**, and building a **chatbot UI with Gradio**.

In [ ]:
# imports

import os
import json
import base64
from io import BytesIO

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from PIL import Image, ImageDraw, ImageFont

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-mini"
IMAGE_MODEL = "gpt-image-1-mini"
openai = OpenAI()

## 1. Content generation (caption + quote + image description)

With a single LLM call, using **structured output** (`json_schema`), we generate three things at once:

- `image_prompt`: an English description to hand to the image generation model
- `caption`: a short, catchy caption to overlay on the image
- `quote`: an inspiring quote from a book
- `quote_source`: the source of the quote (Author — Book Title)

In [ ]:
system_message = """
You are a creative content assistant who prepares inspiring image posts for social media.
The user will give you a topic/theme (e.g. 'solitude', 'hope', 'the ocean', 'new beginnings').

Based on this theme, generate the following:
1. image_prompt: A vivid, artistic image description, written in ENGLISH, for the image generation model.
   - The background, setting, color palette, and art style (e.g. photography, oil painting, watercolor,
     digital art, minimalist, surreal, cinematic) MUST be chosen specifically to fit the meaning of the
     theme, not a generic default.
   - Vary the art style and composition from theme to theme - avoid reusing the same visual formula
     (e.g. don't always default to vibrant pop-art) so that different topics produce visibly different images.
2. caption: A short, catchy caption, in ENGLISH, at most 8-10 words, to overlay on the image.
3. quote: A real quote from a book, in ENGLISH, related to the theme.
4. quote_source: The source of the quote, formatted as 'Author Name — Book Title'.

Always generate valid, theme-appropriate content.
"""

CONTENT_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "content_package",
        "schema": {
            "type": "object",
            "properties": {
                "image_prompt": {"type": "string", "description": "English prompt for the image generation model"},
                "caption": {"type": "string", "description": "Short English caption to overlay on the image"},
                "quote": {"type": "string", "description": "A quote from a book, in English"},
                "quote_source": {"type": "string", "description": "Author — Book name"}
            },
            "required": ["image_prompt", "caption", "quote", "quote_source"],
            "additionalProperties": False
        },
        "strict": True
    }
}

In [ ]:
def generate_content(topic):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": f"Topic: {topic}"}
    ]
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        response_format=CONTENT_SCHEMA
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
# Let's try it out

generate_content("hope")

## 2. Image generation

We use `gpt-image-1-mini` to generate the image from `image_prompt`.

### Price alert: Each image generation costs a few cents - don't go overboard while testing!

In [ ]:
def artist(image_prompt):
    image_response = openai.images.generate(
        model=IMAGE_MODEL,
        prompt=image_prompt,
        size="1024x1024",
        n=1,
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

## 3. Overlaying text on the image

We place the `caption` at the top and the `quote` + `quote_source` at the bottom of the generated image,
on semi-transparent bands, word-wrapped for readability.

In [ ]:
def _load_font(size):
    candidates = [
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf",  # macOS
        "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",  # Linux
        "C:/Windows/Fonts/arialbd.ttf",  # Windows
    ]
    for path in candidates:
        if os.path.exists(path):
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()

In [ ]:
def wrap_text(draw, text, font, max_width):
    words = text.split()
    lines = []
    current = ""
    for word in words:
        trial = f"{current} {word}".strip()
        if draw.textlength(trial, font=font) <= max_width:
            current = trial
        else:
            if current:
                lines.append(current)
            current = word
    if current:
        lines.append(current)
    return lines

In [ ]:
def add_text_overlay(image, caption, quote, quote_source):
    image = image.convert("RGBA")
    width, height = image.size
    overlay = Image.new("RGBA", image.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)

    padding = int(width * 0.05)
    max_width = width - 2 * padding

    caption_font = _load_font(int(width * 0.055))
    quote_font = _load_font(int(width * 0.032))
    source_font = _load_font(int(width * 0.026))

    caption_lines = wrap_text(draw, caption, caption_font, max_width)
    quote_lines = wrap_text(draw, f"“{quote}”", quote_font, max_width)

    line_spacing = 10

    # Top band: caption
    caption_line_height = caption_font.getbbox("Ag")[3] + line_spacing
    caption_block_height = caption_line_height * len(caption_lines) + padding
    draw.rectangle([(0, 0), (width, caption_block_height)], fill=(0, 0, 0, 140))

    y = padding // 2
    for line in caption_lines:
        line_width = draw.textlength(line, font=caption_font)
        draw.text(((width - line_width) / 2, y), line, font=caption_font, fill=(255, 255, 255, 255))
        y += caption_line_height

    # Bottom band: quote + source
    quote_line_height = quote_font.getbbox("Ag")[3] + line_spacing
    source_height = source_font.getbbox("Ag")[3] + line_spacing
    bottom_block_height = quote_line_height * len(quote_lines) + source_height + padding

    draw.rectangle([(0, height - bottom_block_height), (width, height)], fill=(0, 0, 0, 140))

    y = height - bottom_block_height + padding // 2
    for line in quote_lines:
        line_width = draw.textlength(line, font=quote_font)
        draw.text(((width - line_width) / 2, y), line, font=quote_font, fill=(255, 255, 255, 255))
        y += quote_line_height

    source_width = draw.textlength(quote_source, font=source_font)
    draw.text(((width - source_width) / 2, y), quote_source, font=source_font, fill=(230, 230, 230, 255))

    return Image.alpha_composite(image, overlay).convert("RGB")

## 4. Putting it all together

In [ ]:
def create_post(topic):
    content = generate_content(topic)
    image = artist(content["image_prompt"])
    final_image = add_text_overlay(image, content["caption"], content["quote"], content["quote_source"])
    return final_image, content

In [ ]:
# Let's try it out

final_image, content = create_post("new beginnings")
display(final_image)
content

## 5. Gradio chatbot interface

The user types a topic; the chat panel shows the generated caption and quote, while the panel on the
right shows the final image with the text overlaid.

In [ ]:
def chat(history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    topic = history[-1]["content"]

    content = generate_content(topic)
    image = artist(content["image_prompt"])
    final_image = add_text_overlay(image, content["caption"], content["quote"], content["quote_source"])

    reply = (
        f"**Caption:** {content['caption']}\n\n"
        f"**Quote:** \"{content['quote']}\"\n"
        f"— {content['quote_source']}"
    )
    history += [{"role": "assistant", "content": reply}]

    return history, final_image

In [ ]:
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages", label="Caption & Quote")
        image_output = gr.Image(height=500, interactive=False, label="Generated Image")
    with gr.Row():
        message = gr.Textbox(label="Enter a topic/theme (e.g. hope, ocean, solitude):")

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, image_output]
    )

ui.launch(inbrowser=True)

## Ideas / Extensions

- Add a user-selectable image **style** (pop-art, watercolor, minimalist...) as a parameter.
- Integrate a real quote database/search tool (tool calling) to verify `quote_source`.
- Automatically save the generated image to disk and present it in a gallery.
- Change the `size` parameter to support different aspect ratios (Instagram post / story).